In [ ]:
# import
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import json
from datetime import datetime, timedelta
from dateutil import parser

In [2]:
# 설정
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

MAX_PAGES = 5
BASE_DIR = "inven_out"

# 실제 게시판 URL
LOSTARK_MOBILE_URL = "https://www.inven.co.kr/board/lostarkmobile/6389"
AION2_URL = "https://www.inven.co.kr/board/aion2/6388?my=chuchu"

# 아이온2 출시 기준 2주
AION2_RELEASE_DATE = datetime(2025, 11, 19)
AION2_START = AION2_RELEASE_DATE - timedelta(days=14)
AION2_END = AION2_RELEASE_DATE + timedelta(days=14)

def sleep():
    time.sleep(random.uniform(1.0, 2.0))

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def load_state(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"done_urls": [], "data": []}

def save_state(path, state):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

In [3]:
# 링크 수집
def get_post_links(base_url, page):
    url = f"{base_url}?p={page}"
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    links = []
    rows = soup.select("table tbody tr")
    for row in rows:
        a_tag = row.select_one("td.tit a")
        if not a_tag:
            continue
        title = a_tag.text.strip()
        link = a_tag.get("href")
        links.append((title, link))
    return links

# 상세 수집
def get_post_detail(title, url):
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    content_tag = soup.select_one(".contentBody")
    content = content_tag.text.strip() if content_tag else ""

    created_at = ""
    date_tag = soup.select_one(".articleDate")
    if date_tag:
        created_at = date_tag.text.strip()

    views = 0
    view_tag = soup.select_one(".articleHit")
    if view_tag:
        try:
            views = int(view_tag.text.replace(",", "").strip())
        except:
            pass

    likes = 0
    like_tag = soup.select_one("div.recommend span.num")
    if like_tag:
        try:
            likes = int(like_tag.text.strip())
        except:
            pass

    comments = []
    comment_tags = soup.select(".cmtContent")
    for c in comment_tags:
        text = c.text.strip()
        if text:
            comments.append(text)

    return {
        "title": title,
        "content": content,
        "comments": comments,
        "url": url,
        "created_at": created_at,
        "views": views,
        "likes": likes,
        "comment_count": len(comments)
    }

# 크롤링 (중간 저장 + 진행률 표시)
def crawl_inven(base_url, game_type, filter_dates=None):
    game_dir = os.path.join(BASE_DIR, game_type)
    ensure_dir(game_dir)

    state_path = os.path.join(game_dir, "state.json")
    state = load_state(state_path)

    done_urls = set(state["done_urls"])
    data = state["data"]

    all_links = []
    for page in range(1, MAX_PAGES + 1):
        print(f"[{game_type}] page {page}/{MAX_PAGES}")
        links = get_post_links(base_url, page)
        all_links.extend(links)
        sleep()

    total_posts = len(all_links)
    print(f"[{game_type}] total {total_posts} posts")

    for idx, (title, link) in enumerate(all_links, 1):
        if link in done_urls:
            continue

        try:
            post_data = get_post_detail(title, link)

            # 아이온2 날짜 필터
            if filter_dates:
                try:
                    post_dt = datetime.strptime(post_data['created_at'], "%Y-%m-%d %H:%M")
                    if post_dt < filter_dates[0] or post_dt > filter_dates[1]:
                        continue
                except:
                    pass

            # 진행률 표시
            percent = round(idx / total_posts * 100, 1)
            print(f"[posts] {idx}/{total_posts} ({percent}%)")
            print(f"[comments] {link.split('/')[-1]} -> {post_data['comment_count']} comments")

            data.append(post_data)
            done_urls.add(link)

            # 중간저장
            save_state(state_path, {"done_urls": list(done_urls), "data": data})
            sleep()
        except Exception as e:
            print("error:", e)

    return data

# CSV 저장
def save_to_csv(data, game_type):
    game_dir = os.path.join(BASE_DIR, game_type)
    ensure_dir(game_dir)

    today = datetime.now().strftime("%Y%m%d")
    filename = os.path.join(game_dir, f"{game_type}_{today}.csv")

    rows = []
    for post in data:
        if post["comments"]:
            for c in post["comments"]:
                rows.append({
                    "title": post["title"],
                    "content": post["content"],
                    "comment": c,
                    "url": post["url"],
                    "created_at": post["created_at"],
                    "views": post["views"],
                    "likes": post["likes"],
                    "comment_count": post["comment_count"]
                })
        else:
            rows.append({
                "title": post["title"],
                "content": post["content"],
                "comment": "",
                "url": post["url"],
                "created_at": post["created_at"],
                "views": post["views"],
                "likes": post["likes"],
                "comment_count": post["comment_count"]
            })

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved {filename}")

In [18]:
# 실행

# 로스트아크 모바일
lostark_data = crawl_inven(LOSTARK_MOBILE_URL, "lostark_mobile")
save_to_csv(lostark_data, "lostark_mobile")

# 아이온2
aion2_data = crawl_inven(
    AION2_URL,
    "aion2",
    filter_dates=(AION2_START, AION2_END)
)
save_to_csv(aion2_data, "aion2")

print("Done crawling")

[lostark_mobile] page 1/5
[lostark_mobile] page 2/5
[lostark_mobile] page 3/5
[lostark_mobile] page 4/5
[lostark_mobile] page 5/5
[lostark_mobile] total 226 posts
[posts] 1/226 (0.4%)
[comments] 1176 -> 0 comments
[posts] 2/226 (0.9%)
[comments] 1229 -> 0 comments
[posts] 3/226 (1.3%)
[comments] 1197 -> 0 comments
[posts] 4/226 (1.8%)
[comments] 1193 -> 0 comments
[posts] 5/226 (2.2%)
[comments] 1191 -> 0 comments
[posts] 6/226 (2.7%)
[comments] 1190 -> 0 comments
[posts] 7/226 (3.1%)
[comments] 1189 -> 0 comments
[posts] 8/226 (3.5%)
[comments] 1188 -> 0 comments
[posts] 9/226 (4.0%)
[comments] 1187 -> 0 comments
[posts] 10/226 (4.4%)
[comments] 1186 -> 0 comments
[posts] 11/226 (4.9%)
[comments] 1185 -> 0 comments
[posts] 12/226 (5.3%)
[comments] 1184 -> 0 comments
[posts] 13/226 (5.8%)
[comments] 1181 -> 0 comments
[posts] 14/226 (6.2%)
[comments] 1180 -> 0 comments
[posts] 15/226 (6.6%)
[comments] 1179 -> 0 comments
[posts] 16/226 (7.1%)
[comments] 1178 -> 0 comments
[posts] 17/226

In [4]:
import os
import json
import time
import random
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import requests
import pandas as pd
from bs4 import BeautifulSoup

In [11]:
"""===============================================
트러블 슈팅 : 아이온2 데이터 수집 오류 확인
원인 : 날짜 파싱 오류, 웹 크롤링 방식에서 URL 정상적으로 못 가져오는 현상, 인벤 게시판 구조가 다름

requests로 못 가져오고 브라우저 기반 수집(Selenium) 방법 선택
날짜 필터 유연화 (dateutil.parser)
중간 저장 (state.json) 적용
진행률 표시
CSV 저장
MAX_PAGES 확장 (2주치 게시글 확보)
==============================================="""
# import
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

BASE_URL = "https://www.inven.co.kr/board/aion2/6388"
HEADERS = {"User-Agent": "Mozilla/5.0"}

MAX_PAGE = 20
BASE_DIR = "inven_out/aion2"
os.makedirs(BASE_DIR, exist_ok=True)

def get_post_links(page):
    url = f"{BASE_URL}?page={page}"
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    posts = []
    rows = soup.select("tr")

    for row in rows:
        title_tag = row.select_one("td.tit a.subject-link")
        if not title_tag:
            continue

        posts.append({
            "title": title_tag.text.strip(),
            "url": title_tag["href"]
        })

    return posts

def get_post_detail(post):
    res = requests.get(post["url"], headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    content_tag = soup.select_one(".contentBody")
    content = content_tag.text.strip() if content_tag else ""

    # 댓글
    comments = []
    comment_tags = soup.select(".cmtContent")
    for c in comment_tags:
        txt = c.text.strip()
        if txt:
            comments.append(txt)

    return {
        "title": post["title"],
        "content": content,
        "comments": comments,
        "url": post["url"],
        "comment_count": len(comments)
    }

# 전체 수집
all_posts = []

for page in range(1, MAX_PAGE + 1):
    print(f"[게시글] {page}/{MAX_PAGE} 페이지 수집 중 ({round(page/MAX_PAGE*100,1)}%)")

    posts = get_post_links(page)
    if not posts:
        print("게시글 없음 → 종료")
        break

    all_posts.extend(posts)
    time.sleep(1)

print(f"총 게시글 수: {len(all_posts)}")

# 상세 + 댓글 수집
final_data = []

total = len(all_posts)

for idx, post in enumerate(all_posts, 1):
    print(f"[댓글] {idx}/{total} ({round(idx/total*100,1)}%) 수집 중")

    detail = get_post_detail(post)

    if detail["comments"]:
        for c in detail["comments"]:
            final_data.append({
                "title": detail["title"],
                "content": detail["content"],
                "comment": c,
                "url": detail["url"],
                "comment_count": detail["comment_count"]
            })
    else:
        final_data.append({
            "title": detail["title"],
            "content": detail["content"],
            "comment": "",
            "url": detail["url"],
            "comment_count": detail["comment_count"]
        })

    time.sleep(0.5)

# 저장 
df = pd.DataFrame(final_data)

today = pd.Timestamp.now().strftime("%Y%m%d")
filename = os.path.join(BASE_DIR, f"aion2_{today}.csv")

df.to_csv(filename, index=False, encoding="utf-8-sig")

print(f"저장 완료: {filename}")

[게시글] 1/20 페이지 수집 중 (5.0%)
[게시글] 2/20 페이지 수집 중 (10.0%)
[게시글] 3/20 페이지 수집 중 (15.0%)
[게시글] 4/20 페이지 수집 중 (20.0%)
[게시글] 5/20 페이지 수집 중 (25.0%)
[게시글] 6/20 페이지 수집 중 (30.0%)
[게시글] 7/20 페이지 수집 중 (35.0%)
[게시글] 8/20 페이지 수집 중 (40.0%)
[게시글] 9/20 페이지 수집 중 (45.0%)
[게시글] 10/20 페이지 수집 중 (50.0%)
[게시글] 11/20 페이지 수집 중 (55.0%)
[게시글] 12/20 페이지 수집 중 (60.0%)
[게시글] 13/20 페이지 수집 중 (65.0%)
[게시글] 14/20 페이지 수집 중 (70.0%)
[게시글] 15/20 페이지 수집 중 (75.0%)
[게시글] 16/20 페이지 수집 중 (80.0%)
[게시글] 17/20 페이지 수집 중 (85.0%)
[게시글] 18/20 페이지 수집 중 (90.0%)
[게시글] 19/20 페이지 수집 중 (95.0%)
[게시글] 20/20 페이지 수집 중 (100.0%)
총 게시글 수: 1400
[댓글] 1/1400 (0.1%) 수집 중
[댓글] 2/1400 (0.1%) 수집 중
[댓글] 3/1400 (0.2%) 수집 중
[댓글] 4/1400 (0.3%) 수집 중
[댓글] 5/1400 (0.4%) 수집 중
[댓글] 6/1400 (0.4%) 수집 중
[댓글] 7/1400 (0.5%) 수집 중
[댓글] 8/1400 (0.6%) 수집 중
[댓글] 9/1400 (0.6%) 수집 중
[댓글] 10/1400 (0.7%) 수집 중
[댓글] 11/1400 (0.8%) 수집 중
[댓글] 12/1400 (0.9%) 수집 중
[댓글] 13/1400 (0.9%) 수집 중
[댓글] 14/1400 (1.0%) 수집 중
[댓글] 15/1400 (1.1%) 수집 중
[댓글] 16/1400 (1.1%) 수집 중
[댓글] 17/1400 (1.2%) 수집 중

In [14]:
# import
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

BASE_URL = "https://www.inven.co.kr/board/aion2/6388"
HEADERS = {"User-Agent": "Mozilla/5.0"}

MAX_PAGE = 20
BASE_DIR = "inven_out/aion2"
os.makedirs(BASE_DIR, exist_ok=True)

def get_post_links(page):
    url = f"{BASE_URL}?page={page}"
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    posts = []
    rows = soup.select("tr")

    for row in rows:
        title_tag = row.select_one("td.tit a.subject-link")
        if not title_tag:
            continue

        posts.append({
            "title": title_tag.text.strip(),
            "url": title_tag["href"]
        })

    return posts

def get_post_detail(post):
    res = requests.get(post["url"], headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    # 본문 (comment로 쓸 예정)
    content_tag = soup.select_one(".contentBody")
    content = content_tag.text.strip() if content_tag else ""

    # 작성일
    created_at = ""
    date_tag = soup.select_one(".articleDate")
    if date_tag:
        created_at = date_tag.text.strip()

    # 조회수
    views = 0
    view_tag = soup.select_one(".articleHit")
    if view_tag:
        try:
            views = int(view_tag.text.replace(",", "").strip())
        except:
            pass

    # 좋아요
    likes = 0
    like_tag = soup.select_one("div.recommend span.num")
    if like_tag:
        try:
            likes = int(like_tag.text.strip())
        except:
            pass

    # 댓글
    comments = []
    comment_tags = soup.select(".cmtContent")
    for c in comment_tags:
        txt = c.text.strip()
        if txt:
            comments.append(txt)

    return {
        "title": post["title"],
        "content": content,
        "comments": comments,
        "url": post["url"],
        "created_at": created_at,
        "views": views,
        "likes": likes,
        "comment_count": len(comments)
    }

# 전체 수집
all_posts = []

for page in range(1, MAX_PAGE + 1):
    print(f"[게시글] {page}/{MAX_PAGE} ({round(page/MAX_PAGE*100,1)}%)")

    posts = get_post_links(page)
    if not posts:
        print("게시글 없음 → 종료")
        break

    all_posts.extend(posts)
    time.sleep(1)

print(f"총 게시글 수: {len(all_posts)}")

# 상세 + 댓글 수집
final_data = []
total = len(all_posts)

for idx, post in enumerate(all_posts, 1):
    print(f"[댓글] {idx}/{total} ({round(idx/total*100,1)}%)")

    detail = get_post_detail(post)

    # 본문을 comment로 사용
    if detail["comments"]:
        for c in detail["comments"]:
            final_data.append({
                "title": detail["title"],
                "comment": c,  # 댓글
                "post_content": detail["content"],  # 본문 따로 유지 (추천)
                "url": detail["url"],
                "created_at": detail["created_at"],
                "views": detail["views"],
                "likes": detail["likes"],
                "comment_count": detail["comment_count"]
            })
    else:
        final_data.append({
            "title": detail["title"],
            "comment": "",  
            "post_content": detail["content"],
            "url": detail["url"],
            "created_at": detail["created_at"],
            "views": detail["views"],
            "likes": detail["likes"],
            "comment_count": detail["comment_count"]
        })

    time.sleep(0.5)

# 저장
df = pd.DataFrame(final_data)

today = pd.Timestamp.now().strftime("%Y%m%d")
filename = os.path.join(BASE_DIR, f"aion2_{today}.csv")

df.to_csv(filename, index=False, encoding="utf-8-sig")

print(f"저장 완료: {filename}")

[게시글] 1/20 (5.0%)
[게시글] 2/20 (10.0%)
[게시글] 3/20 (15.0%)
[게시글] 4/20 (20.0%)
[게시글] 5/20 (25.0%)
[게시글] 6/20 (30.0%)
[게시글] 7/20 (35.0%)
[게시글] 8/20 (40.0%)
[게시글] 9/20 (45.0%)
[게시글] 10/20 (50.0%)
[게시글] 11/20 (55.0%)
[게시글] 12/20 (60.0%)
[게시글] 13/20 (65.0%)
[게시글] 14/20 (70.0%)
[게시글] 15/20 (75.0%)
[게시글] 16/20 (80.0%)
[게시글] 17/20 (85.0%)
[게시글] 18/20 (90.0%)
[게시글] 19/20 (95.0%)
[게시글] 20/20 (100.0%)
총 게시글 수: 1400
[댓글] 1/1400 (0.1%)
[댓글] 2/1400 (0.1%)
[댓글] 3/1400 (0.2%)
[댓글] 4/1400 (0.3%)
[댓글] 5/1400 (0.4%)
[댓글] 6/1400 (0.4%)
[댓글] 7/1400 (0.5%)
[댓글] 8/1400 (0.6%)
[댓글] 9/1400 (0.6%)
[댓글] 10/1400 (0.7%)
[댓글] 11/1400 (0.8%)
[댓글] 12/1400 (0.9%)
[댓글] 13/1400 (0.9%)
[댓글] 14/1400 (1.0%)
[댓글] 15/1400 (1.1%)
[댓글] 16/1400 (1.1%)
[댓글] 17/1400 (1.2%)
[댓글] 18/1400 (1.3%)
[댓글] 19/1400 (1.4%)
[댓글] 20/1400 (1.4%)
[댓글] 21/1400 (1.5%)
[댓글] 22/1400 (1.6%)
[댓글] 23/1400 (1.6%)
[댓글] 24/1400 (1.7%)
[댓글] 25/1400 (1.8%)
[댓글] 26/1400 (1.9%)
[댓글] 27/1400 (1.9%)
[댓글] 28/1400 (2.0%)
[댓글] 29/1400 (2.1%)
[댓글] 30/1400 (2.1%)
[댓글]

In [23]:
# 기존 댓글 컬럼 삭제 (있을 경우만)
cols_to_drop = [col for col in ["comment", "comment_count"] if col in aion_df.columns]
aion_df = aion_df.drop(columns=cols_to_drop)

In [32]:
# content → comment로 이름 변경
aion_df = aion_df.rename(columns={"comment": "content"})


In [33]:
aion_df

,title,content,url,created_at,views,likes
0,[잡담]\n아이온2 인장 캐릭터 인증 가이드,아이온2 인벤에 아이온2 인장 연동 기능이 추가되었습니다. 아래 안내를 따라와 주시...,https://www.inven.co.kr/board/aion2/6388/72656,2026-01-21 12:19,0,0
1,[소식]\n ...,쿠폰 입력 기간 변경 [바로가기]시즌2 일정 변경에 따라 AION2 웰컴백 쿠폰 &...,https://www.inven.co.kr/board/aion2/6388/104900,2026-02-23 19:31,0,0
2,"[잡담]\n전투력 업데이트 완료! 인장 갱신하면 3,000 이니 드려요!","안녕하세요, 아이온2 인벤팀입니다.아이온2 인증 인장에 전투력 수치가 추가되었습니다...",https://www.inven.co.kr/board/aion2/6388/125736,2026-03-19 17:08,0,0
3,[소식]\n ...,[콘텐츠 및 시스템]어비스 매칭 개편어비스 매칭에서 동일 종족도 적대 종족으로 매칭...,https://www.inven.co.kr/board/aion2/6388/131176,2026-03-25 10:23,0,0
4,[소식]\n ...,"두 개의 하늘, 하나의 영광안녕하세요, 데바 여러분. AION2입니다.​항상 AIO...",https://www.inven.co.kr/board/aion2/6388/131177,2026-03-25 10:24,0,0
...,...,...,...,...,...,...
1395,[잡담]\n ...,임점후 수호성 마도 정령성 너프 검성 전체스킬 딜상향 줌 <<<<<<<<<<<수호성...,https://www.inven.co.kr/board/aion2/6388/134053,2026-03-26 13:09,0,0
1396,[잡담]\n ...,얼릉 팔어,https://www.inven.co.kr/board/aion2/6388/134052,2026-03-26 13:09,0,0
1397,[잡담]\n ...,pc로 한다고 pc게임이겠냐호박에 줄긋는다고 수박되냐고 ㅋㅋㅋ직업간 밸런스에서 정통...,https://www.inven.co.kr/board/aion2/6388/134051,2026-03-26 13:09,0,0
1398,[잡담]\n ...,그냥 갑자기 든 생각수호성은 딴딴한 이미지검성은 딜탱 하이브리드 스위치 온오프 만능...,https://www.inven.co.kr/board/aion2/6388/134050,2026-03-26 13:09,0,0


In [35]:
# 한 파일로 합치기

# 파일 경로
aion_path = "inven_out/aion2/aion2_20260326.csv"
lostark_path = "inven_out/lostark_mobile/lostark_mobile_20260325.csv"

# 불러오기
aion_df = pd.read_csv(aion_path)
lostark_df = pd.read_csv(lostark_path)

# 컬럼 추가
aion_df["source"] = "inven_aion2"
lostark_df["source"] = "inven_lostarkmobile"

# 공통 region
aion_df["region"] = "국내"
lostark_df["region"] = "국내"

# 컬럼 구조 맞추기 (혹시 컬럼 다를 경우 대비)
common_cols = list(set(aion_df.columns) | set(lostark_df.columns))

aion_df = aion_df.reindex(columns=common_cols)
lostark_df = lostark_df.reindex(columns=common_cols)

# 합치기
merged_df = pd.concat([aion_df, lostark_df], ignore_index=True)

print("합치기 완료")
print("총 데이터 수:", len(merged_df))
print("컬럼:", merged_df.columns.tolist())

합치기 완료
총 데이터 수: 1625
컬럼: ['region', 'views', 'title', 'created_at', 'content', 'source', 'comment_count', 'likes', 'comment', 'url']


In [36]:
merged_df

,region,views,title,created_at,content,source,comment_count,likes,comment,url
0,국내,0,[잡담]\n아이온2 인장 캐릭터 인증 가이드,2026-01-21 12:19,아이온2 인벤에 아이온2 인장 연동 기능이 추가되었습니다. 아래 안내를 따라와 주시...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/72656
1,국내,0,[소식]\n ...,2026-02-23 19:31,쿠폰 입력 기간 변경 [바로가기]시즌2 일정 변경에 따라 AION2 웰컴백 쿠폰 &...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/104900
2,국내,0,"[잡담]\n전투력 업데이트 완료! 인장 갱신하면 3,000 이니 드려요!",2026-03-19 17:08,"안녕하세요, 아이온2 인벤팀입니다.아이온2 인증 인장에 전투력 수치가 추가되었습니다...",inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/125736
3,국내,0,[소식]\n ...,2026-03-25 10:23,[콘텐츠 및 시스템]어비스 매칭 개편어비스 매칭에서 동일 종족도 적대 종족으로 매칭...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/131176
4,국내,0,[소식]\n ...,2026-03-25 10:24,"두 개의 하늘, 하나의 영광안녕하세요, 데바 여러분. AION2입니다.​항상 AIO...",inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/131177
...,...,...,...,...,...,...,...,...,...,...
1620,국내,0,[잡담]\n ...,2025-11-14 16:47,소환사 제대로 나와서 즐길수 있었으면 좋겠어요,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1621,국내,0,[잡담]\n ...,2025-11-14 16:42,맞으며,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1622,국내,0,[잡담]\n ...,2025-11-14 15:55,일단 나,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1623,국내,0,[잡담]\n ...,2025-11-14 15:43,기대중입니다,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...


In [38]:
import re

def contains_link(text: str) -> bool:
    if pd.isna(text):
        return False
    return bool(re.search(r"http[s]?://\S+", str(text)))

# 링크가 있는 행 삭제
merged_df = merged_df[~(merged_df["title"].apply(contains_link) | merged_df["content"].apply(contains_link))].copy()

In [39]:
merged_df

,region,views,title,created_at,content,source,comment_count,likes,comment,url
0,국내,0,[잡담]\n아이온2 인장 캐릭터 인증 가이드,2026-01-21 12:19,아이온2 인벤에 아이온2 인장 연동 기능이 추가되었습니다. 아래 안내를 따라와 주시...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/72656
1,국내,0,[소식]\n ...,2026-02-23 19:31,쿠폰 입력 기간 변경 [바로가기]시즌2 일정 변경에 따라 AION2 웰컴백 쿠폰 &...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/104900
2,국내,0,"[잡담]\n전투력 업데이트 완료! 인장 갱신하면 3,000 이니 드려요!",2026-03-19 17:08,"안녕하세요, 아이온2 인벤팀입니다.아이온2 인증 인장에 전투력 수치가 추가되었습니다...",inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/125736
3,국내,0,[소식]\n ...,2026-03-25 10:23,[콘텐츠 및 시스템]어비스 매칭 개편어비스 매칭에서 동일 종족도 적대 종족으로 매칭...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/131176
4,국내,0,[소식]\n ...,2026-03-25 10:24,"두 개의 하늘, 하나의 영광안녕하세요, 데바 여러분. AION2입니다.​항상 AIO...",inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/131177
...,...,...,...,...,...,...,...,...,...,...
1619,국내,0,[잡담]\n ...,2025-11-14 16:49,넘 기대됩니다,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1620,국내,0,[잡담]\n ...,2025-11-14 16:47,소환사 제대로 나와서 즐길수 있었으면 좋겠어요,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1621,국내,0,[잡담]\n ...,2025-11-14 16:42,맞으며,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1622,국내,0,[잡담]\n ...,2025-11-14 15:55,일단 나,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...


In [41]:
merged_df = merged_df.rename(columns={"content": "selftext"})

In [46]:
merged_df

,region,views,title,created_at,selftext,source,comment_count,likes,comment,url
0,국내,0,[잡담]\n아이온2 인장 캐릭터 인증 가이드,2026-01-21 12:19,아이온2 인벤에 아이온2 인장 연동 기능이 추가되었습니다. 아래 안내를 따라와 주시...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/72656
1,국내,0,[소식]\n ...,2026-02-23 19:31,쿠폰 입력 기간 변경 [바로가기]시즌2 일정 변경에 따라 AION2 웰컴백 쿠폰 &...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/104900
2,국내,0,"[잡담]\n전투력 업데이트 완료! 인장 갱신하면 3,000 이니 드려요!",2026-03-19 17:08,"안녕하세요, 아이온2 인벤팀입니다.아이온2 인증 인장에 전투력 수치가 추가되었습니다...",inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/125736
3,국내,0,[소식]\n ...,2026-03-25 10:23,[콘텐츠 및 시스템]어비스 매칭 개편어비스 매칭에서 동일 종족도 적대 종족으로 매칭...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/131176
4,국내,0,[소식]\n ...,2026-03-25 10:24,"두 개의 하늘, 하나의 영광안녕하세요, 데바 여러분. AION2입니다.​항상 AIO...",inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/131177
...,...,...,...,...,...,...,...,...,...,...
1619,국내,0,[잡담]\n ...,2025-11-14 16:49,넘 기대됩니다,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1620,국내,0,[잡담]\n ...,2025-11-14 16:47,소환사 제대로 나와서 즐길수 있었으면 좋겠어요,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1621,국내,0,[잡담]\n ...,2025-11-14 16:42,맞으며,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1622,국내,0,[잡담]\n ...,2025-11-14 15:55,일단 나,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...


In [50]:
# 아이온2 - [잡담] 포함된 데이터만 필터링
aion_filtered = merged_df[
    (merged_df["source"] == "inven_aion2") &
    (merged_df["title"].str.contains(r"\[잡담\]", na=False))
]

# 로스트아크모바일 그대로 유지
lostark_df = merged_df[merged_df["source"] == "inven_lostarkmobile"]

# 다시 합치기
final_df = pd.concat([aion_filtered, lostark_df], ignore_index=True)
final_df

,region,views,title,created_at,selftext,source,comment_count,likes,comment,url
0,국내,0,[잡담]\n아이온2 인장 캐릭터 인증 가이드,2026-01-21 12:19,아이온2 인벤에 아이온2 인장 연동 기능이 추가되었습니다. 아래 안내를 따라와 주시...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/72656
1,국내,0,"[잡담]\n전투력 업데이트 완료! 인장 갱신하면 3,000 이니 드려요!",2026-03-19 17:08,"안녕하세요, 아이온2 인벤팀입니다.아이온2 인증 인장에 전투력 수치가 추가되었습니다...",inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/125736
2,국내,0,[잡담]\n ...,2026-03-26 13:30,우쮸쮸 끝판왕이네 ㅋㅋ죽는거무섭다니까무한부활줘버리네부활석은 쓰냐?치유들이 잘도 좋아...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/134112
3,국내,0,[잡담]\n ...,2026-03-26 13:30,일주일도 안돼서 애가 저렇게 타락함?,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/134111
4,국내,0,[잡담]\n ...,2026-03-26 13:29,궁성 탱인데 딜좀 너프해야되는거아님??피흡으로 다버티는데 아이온2는 원거리탱이라는 ...,inven_aion2,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/134110
...,...,...,...,...,...,...,...,...,...,...
1497,국내,0,[잡담]\n ...,2025-11-14 16:49,넘 기대됩니다,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1498,국내,0,[잡담]\n ...,2025-11-14 16:47,소환사 제대로 나와서 즐길수 있었으면 좋겠어요,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1499,국내,0,[잡담]\n ...,2025-11-14 16:42,맞으며,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...
1500,국내,0,[잡담]\n ...,2025-11-14 15:55,일단 나,inven_lostarkmobile,0.0,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...


In [51]:
# 저장
final_df.to_csv("inven_all_posts.csv", index=False, encoding="utf-8-sig")